# Data Preprocessing Notes

## Topic 1: Pandas Fundamentals

**What is pandas?** 

Pandas is a python library built for data manipulation and analysis. It gives you two main data structures. Series (1D) and DataFrame (2D, like a table/spreadsheet)

**1. Key Concepts**

```python
# installing and importing
import pandas as pd

# loading a csv file 
df = pd.read_csv('dataset.csv')
```

**2. Viewing your data**

```python
df.head()       # first 5 rows
df.tail()       # last 5 rows
df.shape        # (rows, columns)
df.columns      # list of column names
df.dtypes       # data type of each column
```

**3. Selecting columns**

```python
df['Age']              # single column → returns a Series
df[['Age', 'Fare']]   # multiple columns → returns a DataFrame
```

**4. Selecting rows**

```python
df.iloc[0]        # by index position (first row)
df.loc[0]         # by label/index
df[df['Age'] > 30]  # conditional filtering
```

**5. Basic Operations**

```python
df['Age'].mean()      # average
df['Age'].median()    # median
df['Age'].value_counts()  # frequency count
df['Fare'].sum()      # total
```

**6. Adding a new column**

```python
df['Age_x2'] = df['Age'] * 2
```

**7. Dropping a column**

```python
df.drop('Age_x2', axis=1, inplace=True)
```

- axis=1 means column, axis=0 means row 
- inplace=True modifies the original DataFrame 

**8. Sorting**

```python
df.sort_values('Age', ascending=False)
```

**9. Handling duplicates**

```python
df.duplicated().sum()    # count duplicates
df.drop_duplicates()     # remove them
```

## Topic 2: Exploratory Data Analysis (EDA)

**What is EDA?**

EDA is the process of understanding your data before doing anything with it. You look at patterns, distributions, relationships, and anomalies. Think of it as getting to know your dataset before building any model.

**1. Key Concepts**

```python
df.describe()
# this gives you count, mean, std, min, Q1, Q2, Q3, max for al numerical columns. 
```

```python
df.info()
# This tells you column names, non-null counts, and data types. Super useful to spot missing values quickly.
```

**2. Checking Missing Values**

```python
df.isnull().sum()
# shows how many missing values each column has
```

```python
df.isnull().sum() / len(df) * 100
# Shows missing values as a percentage — more intuitive.
```

**3. Visualizing Distributions**

```python
import matplotlib.pyplot as plt
import seaborn as sns
```

```python
# histogram - for numerical columns
df['Age'].hist(bins=30)
plt.xlabel('Age')
plt.ylabel('Count')
plt.title('Age Distribution')
plt.show()
# Tells you if the data is skewed, has gaps, or clusters around certain values.
```

```python
# countplot - for categorical columns
sns.countplot(x='Survived', data=df)
plt.show()
# Shows frequency of each category. Here you can quickly see how many survived vs didn't.
```

```python
# boxplot - for spotting outliers 
sns.boxplot(x=df['Fare'])
plt.show()
# the dots beyond the whiskers are outliers
```

**4. Relationships Between Variables (Bivariate Analysis)**

This is about understanding how two variables relate. 

```python
# countplot with hue
sns.countplot(x='Survived', hue='Sex', data=df)
plt.show()
# This shows survival count split by gender. 
```

```python
# boxplot with grouping 
sns.boxplot(x='Survived', y='Age', data=df)
plt.show()
# Shows how age distribution differs between survivors and non-survivors.
```

```python
# scatterplot
sns.scatterplot(x='Age', y='Fare', hue='Survived', data=df)
plt.show()
# Helps you see if there's a pattern between two numerical features.
```

**5. Correlation Matrix (Multivariate Analysis)**

```python
df.corr(numeric_only=True)
# Shows how strongly numerical features are related to each other. Values range from -1 to 1.
```

```python
# heatmap - visualize the correlation matrix 
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()
```

- Values close to 1 -> strong positive correlation
- Values close to -1 -> strong negative correlation 
- Values close to 0 -> no correlation

## Topic 3: Handling Missing Values 

**What are missing values?**

Missing values are empty cells in your dataset. In Pandas, they show up as NaN (Not a Number) or None. Almost every real-world dataset has them.

**1. Key Concepts**

```python
df.isnull().sum()

df.isnull().sum() / len(df) * 100 # this gives the percentage
```

**2. Visualizing Missing Values**

```python
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()
# yellow streaks show where data is missing. 
```

**3. Strategies to Handle Missing Values**

<u>Strategy 1: Dropping</u>
```python
df.dropna()  # drop rows with missing values 
df.drop('Cabin', axis=1, inplace=True) # drop a specific column
```

<u>Strategy 2: Imputation (Filling): it means replacing NaN with a value</u>
```python
df['Age'].fillna(df['Age'].mean(), inplace=True) # fill with mean
df['Age'].fillna(df['Age'].median(), inplace=True) # fill with median 
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True) # fill with mode 
df['Cabin'].fillna('Unknown', inplace=True) # fill with a constant
```

**4. Smart Imputation (Group Based)**

```python
# Instead of filling Age with the overall median, you can fill it based on groups. This fills missing Age values with the median age of their passenger class.
df['Age'] = df.groupby('Pclass')['Age'].transform(
    lambda x: x.fillna(x.median())
)
```

## Topic 4: Outlier Detection & Treatment

**What are Outliers?**

Outliers are data points that are significantly different from the rest of the data. For example, if most passengers paid a fare between $5–$50, but one paid $512, that's an outlier.

**1. Visualizing Outliers**

```python
# boxplot
sns.boxplot(x=df['Fare'])
plt.title('Fare Boxplot')
plt.show()

# boxplot for multiple columns
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.boxplot(x=df['Age'], ax=axes[0])
sns.boxplot(x=df['Fare'], ax=axes[1])
sns.boxplot(x=df['SibSp'], ax=axes[2])
plt.tight_layout()
plt.show()

# histogram
df['Fare'].hist(bins=50)
plt.title('Fare Distribution')
plt.show()
```

**2. Detection Methods** 

<u>Method 1: IQR (Interquartile Range) Method </u>
```python
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
```

How it works:

- Q1 = 25% (bottom of the box)
- Q3 = 75% (top of the box)
- IQR = Q3 - Q1 (the height of the box)
- Anything below Q1 - 1.5 * IQR or above Q3 + 1.5 * IQR is an outlier

```python
# finding outliers
outliers = df[(df['Fare'] < lower_bound) | (df['Fare'] > upper_bound)]
print(f'Number of outliers: {len(outliers)}')

# making it reusable
def find_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return outliers, lower, upper
```

<u>Method 2: Z-Score Method</u>
```python
from scipy import stats
z_scores = stats.zscore(df['Fare'].dropna())
outliers = (z_scores > 3) | (z_scores < -3)
print(f'Number of outliers: {outliers.sum()}')
```

How it works:

- Z-score = (value - mean) / std 
- If z-score > 3 or < -3, it's an outlier 
- This assumes data is roughly normally distributed

**3. Treatment Methods**

Once you have found outliers, you have 4 options. 

<u>Option 1: Remove Them</u>
```python
outliers, lower, upper = find_outliers_iqr(df, 'Fare')
df = df[(df['Fare'] >= lower) & (df['Fare'] <= upper)]
```

<u>Option 2: Cap Them (Capping/Winsorization)</u>

Replace outliers with the boundary values. This is the common approach. 
```python
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df['Fare'] = df['Fare'].clip(lower=lower, upper=upper)
```

<u>Option 3: Log Transformation</u>

Compresses the range of data, reducing the impact of outliers.
```python
import numpy as np
df['Fare_log'] = np.log1p(df['Fare'])  # log1p = log(1 + x), handles 0 values
```

## Topic 5: Encoding Categorical Variables

**What is encoding?**

Encoding is the process of converting categorical (text) data into numbers. Machine learning algorithms work with numbers, not strings like "male", "female", or "S", "C", "Q".

**1. Key Concepts**
```python
# identify your categorical columns 
df.select_dtypes(include='object').columns
# this gives you count
df['Sex'].value_counts()
```

**2. Types Of Categorical Data**

<u>Nominal:</u> no order, no ranking

<u>Ordinal:</u> has a natural order/ranking

**3. Encoding Methods**

<u>Method 1: Label Encoding</u> assigns a unique integer to each category
```python
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Sex_encoded'] = le.fit_transform(df['Sex'])
```
```
male   → 1
female → 0
```
When to use: Only for ordinal data where order matters, or for binary columns (2 values only, like Sex).

<u>Method 2: One Hot Encoding (OHE)</u> creates a new binary column for each category
```python
pd.get_dummies(df['Embarked'], prefix='Embarked')
```
```
Embarked_C  Embarked_Q  Embarked_S
    0           0           1
    1           0           0
    0           1           0
```

Using it on the DataFrame 
```python
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
# drop_first=True drops the first category to avoid the dummy variable trap
# If you know Embarked_Q=0 and Embarked_S=0, then it must be Embarked_C. So you don't need all three columns.
```

Using sklearn
```python
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first', sparse_output=False)
encoded = ohe.fit_transform(df[['Embarked']])
```
When to use: for nominal data with no natural order. This is the most common method. 

<u>Method 3: Ordinal Encoding</u> assigns integers based on a specific order you define. 
```python
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(categories=[['3', '2', '1']])  # worst to best
df['Pclass_encoded'] = oe.fit_transform(df[['Pclass']].astype(str))
```
```
3rd class → 0
2nd class → 1
1st class → 2
```
When to use: when the categories have a meaningful order. 

<u>Method 4: Frequency Encoding</u> replace each category with its frequency (count) in the dataset. 
```python
freq = df['Embarked'].value_counts()
df['Embarked_freq'] = df['Embarked'].map(freq)
```
```
S → 644
C → 168
Q →  77
```
When to use: For high cardinality columns where OHE would create too many columns. It preserves some information about the category without exploding dimensions.

## Topic 6: Feature Engneering

**What is feature engineering?**

Feature engineering is the process of creating new meaningful features from existing data to help your model perform better. It's about extracting hidden information that the raw data doesn't directly show.

The mindset: Look at your data and ask - "What information is hidden here that could help predict survival?"

**1. Feature Engineering Techniques**

<u>a. Extracting Title from Name:</u> the name column looks useless at first 
```python
Braund, Mr. Owen Harris
Cumings, Mrs. John Bradley
Heikkinen, Miss. Laina
```
But notice the Title - Mr, Mrs, Miss. This tells you gender, martial status and sometimes social status
```python
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.')
```
This regex extracts the word before the period.
```python
df['Title'].value_counts()
```
```
Mr        517
Miss      182
Mrs       125
Master     40
Dr          7
Rev         6
Col         2
Major       2
Mlle        2
...
```
Group rares title together
```python
df['Title'] = df['Title'].replace(['Dr', 'Rev', 'Col', 'Major', 'Capt', 
                                    'Jonkheer', 'Don', 'Sir', 'Countess', 
                                    'Lady', 'Dona'], 'Rare')
df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')
```
Now you have clean categories Mr, Miss, Mrs, Master, Rare
```python
df['Title'].value_counts()
```
```
Mr        517
Miss      184
Mrs       126
Master     40
Rare       24
```

<u>b. Family Size:</u> SibSp = siblings/spouse count, Parch = parents/children count. Combine them.
```python
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 for the passenger themselves
```
Why? Solo travelers and very large families had lower survival rates. Medium-sized families survived more. This single feature captures that pattern better than SibSp and Parch separately.

<u>c. IsAlone:</u> A binary feature derived from FamilySize
```python
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
```
```
Alone     → 1
Not alone → 0
```
Why? Solo travelers had significantly lower survival. This gives the model a simple yes/no signal.

<u>d. Age Binning:</u> Instead of using exact age, group passengers into age groups.
```python
df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], 
                       labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior'])
```
```
0-12   → Child
13-18  → Teen
19-35  → Young Adult
36-60  → Adult
61-100 → Senior
```

<u>e. Fare Binning:</u> Same idea as age binning, group fares into categories.
```python
df['FareBin'] = pd.qcut(df['Fare'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])
```
- pd.qcut creates bins with equal number of passengers in each bin 
- pd.cut creates bins with equal width ranges

<u>f. Cabin Deck (if you kept Cabin):</u> If you didn't drop Cabin earlier, you can extract the deck letter. 
```python
df['Deck'] = df['Cabin'].str[0]  # first character
```
```
C85  → C
B28  → B
NaN  → NaN
```

<u>g. Fare Per Person:</u> If a family bought tickets together, the Fare might be the total for the whole group.
```python
df['FarePerPerson'] = df['Fare'] / df['FamilySize']
```

<u>h. Interaction Features:</u> Combine two features to create a new one. 
```python
df['Pclass_Sex'] = df['Pclass'].astype(str) + '_' + df['Sex'].astype(str)
```
```
1_female  → highest survival
3_male    → lowest survival
```
Why? A 1st class female had a very different survival chance than a 3rd class male. This combined feature captures that interaction directly.

<u>i. Polynomial Features (Advanced):</u> Create squared or multiplied versions of numerical features.
```python
df['Age_squared'] = df['Age'] ** 2
df['Age_x_Fare'] = df['Age'] * df['Fare']
```
Why? Sometimes the relationship between a feature and the target is non-linear. Polynomial features help linear models capture that.

## Topic 7: Feature Scaling

Feature scaling is the process of bringing all numerical features to a similar range/scale. Right now your features might look like this:
```python
Age:   0 to 80
Fare:  0 to 512
Pclass: 1 to 3
IsAlone: 0 to 1
```
These are on completely different scales. Feature scalling fixes that. 

**When is scaling needed vs not needed?**
| Model / Technique        | Needs Scaling | Doesn’t Need Scaling |
|---------------------------|--------------|----------------------|
| Linear Regression         | ✅           | ❌                   |
| Logistic Regression       | ✅           | ❌                   |
| KNN                       | ✅           | ❌                   |
| SVM                       | ✅           | ❌                   |
| Neural Networks           | ✅           | ❌                   |
| PCA                       | ✅           | ❌                   |
| Decision Trees            | ❌           | ✅                   |
| Random Forest             | ❌           | ✅                   |
| XGBoost / LightGBM        | ❌           | ✅                   |
| Naive Bayes               | ❌*          | ✅                   |

**Scaling Methods**

<u>Method 1: StandardScaler (Z-score Normalization)</u>

Transforms data so that it has mean=0 and standard deviation=1 

Formula:
```python
X_scaled = (X - mean) / std # formula

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])
```
**When to use**

- When your data is roughly normally distributed
- When you don't need values in a specific range
- Most commonly used scaler in practice

<u>Method 2: MinMaxScaler</u>

Transforms data to a fixed range usually 0 to 1. 

Formula:
```python
X_scaled = (X - X_min) / (X_max - X_min)

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])
```

**When to use**

- When you need values in a specific range (0 to 1)
- Neural networks often prefer this
- When your data is not normally distributed

<u>Method 3: RobustScaler</u>

Uses median and IQR instead of mean and std. Robust to outliers. 

Formula:
```python
X_scaled = (X - median) / IQR

from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
df[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])
```

**When to use**

- When your data has significant outliers that you don't want to remove 
- When StandardScaker is affected by extreme values

**Critical Rule:** fit_transform vs transform
```python
# On training data → fit_transform
X_train_scaled = scaler.fit_transform(X_train)
# On test data → ONLY transform
X_test_scaled = scaler.transform(X_test)
```

**Which columns to scale?** scale numerical features, don;t scale binary features. 
```python
numerical_cols = ['Age', 'Fare', 'FamilySize', 'FarePerPerson']
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
```

**Visualizing the effect**
```python
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Before scaling
axes[0].hist(df_original['Age'], bins=30, alpha=0.7, label='Age')
axes[0].hist(df_original['Fare'], bins=30, alpha=0.7, label='Fare')
axes[0].set_title('Before Scaling')
axes[0].legend()
# After scaling
axes[1].hist(df_scaled['Age'], bins=30, alpha=0.7, label='Age')
axes[1].hist(df_scaled['Fare'], bins=30, alpha=0.7, label='Fare')
axes[1].set_title('After StandardScaler')
axes[1].legend()
plt.tight_layout()
plt.show()
```

## Topic 8: Normalization

**What is normalization?**

Normalization is the process of transforming skewed data into a more symmetric, bell-shaped distribution. It's about fixing the shape of your data, not just the range.

**1. How To Check if Data is Skewed**
```python
import matplotlib.pyplot as plt
import seaborn as sns
sns.histplot(df['Fare'], kde=True)
plt.title('Fare Distribution')
plt.show()
# if the curve has a long tail on one side, its skewed
```

```python
# numerical check
df['Fare'].skew()
```
- Skewness = 0 -> perfectly symmetric
- Skewness > 0 → right-skewed (long tail on right)
- Skewness < 0 → left-skewed (long tail on left)
- |Skewness| > 1 → heavily skewed, needs transformation

**2. Normalization Techniques**

<u>Technique 1: Log Transformation</u> the most commonly used normalization technique
```python
import numpy as np
df['Fare_log'] = np.log1p(df['Fare'])  # log(1 + x)
print(f"Before - Skewness: {df['Fare'].skew():.2f}")
print(f"After  - Skewness: {df['Fare_log'].skew():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(df['Fare'], kde=True, ax=axes[0])
axes[0].set_title(f"Original Fare (skew={df['Fare'].skew():.2f})")

sns.histplot(df['Fare_log'], kde=True, ax=axes[1])
axes[1].set_title(f"Log Fare (skew={df['Fare_log'].skew():.2f})")

plt.tight_layout()
plt.show()
```

<u>Technique 2: Square Root Transformation</u> a milder transformation than log
```python
df['Fare_sqrt'] = np.sqrt(df['Fare'])
print(f"Sqrt Skewness:      {df['Fare_sqrt'].skew():.2f}")
# use when data is moderately skewed and log transformation is too aggressive.
```

<u>Technique 3: Box Cox Transformation</u> an automatic transformation that finds the best power transformation for your data.
```python
from scipy import stats
df['Fare_boxcox'], lambda_value = stats.boxcox(df['Fare'] + 1)  # +1 to handle zeros
print(f"Lambda: {lambda_value:.2f}")
print(f"Skewness: {df['Fare_boxcox'].skew():.2f}")
```

**How it works**
- Box-Cox finds the optimal lambda value
- If lambda = 0 → it's equivalent to log transformation
- If lambda = 0.5 → it's equivalent to square root
- If lambda = 1 → no transformation needed

<u>Technique 4: Yeo-Johnson Transformation</u> Like Box-Cox but works with zero and negative values too.
```python
from sklearn.preprocessing import PowerTransformer
pt = PowerTransformer(method='yeo-johnson')
df['Fare_yj'] = pt.fit_transform(df[['Fare']])
# use when your data contains zeros or negative values and box-cox does not work
```

## Topic 9: Building Reusable Pipelines

**What is a pipeline?**

A pipeline is a class or function that chains all your preprocessing steps together in the correct order. Instead of writing 50 lines of code every time you get new data, you call one function and it does everything — cleaning, encoding, engineering, normalizing, scaling — automatically.

**The Correct Order Of Operations**

1. Load data 
2. Drop useless columns 
3. Feature engineering (create new features from raw data)
4. Handle missing values
5. Encode categorical variables 
6. Handle outliers 
7. Normalize skewed features 
8. Scale numerical features

**Design Principles**

1. Always work on a copy
2. Each step is a separate method: Makes it easy to debug. If encoding breaks, you know exactly where to look.
3. The class stores fitted state: The scaler (and any other learned statistics) live inside the class. So when you call transform() later, it uses the same parameters.
4. Handle edge cases